In [7]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
from deepface import DeepFace
from tqdm import tqdm
import subprocess
import os

# Функция для анализа эмоций с помощью DeepFace
def analyze_emotion(frame):
    try:
        # Преобразуем кадр в формат numpy array (BGR), если он не является таковым
        if not isinstance(frame, np.ndarray):
            frame = np.array(frame)
        
        analysis = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
        if isinstance(analysis, list):
            analysis = analysis[0]  # Если анализ возвращает список, берем первый элемент
        emotions = analysis.get('emotion', {})
        region = analysis.get('region', {})
        return emotions, region
    except Exception as e:
        return None, None

# Декодирование видео с помощью ffmpeg и запись кадров во временную папку
video_path = 'video.mp4'  # Укажи путь к видеофайлу
temp_frames_folder = 'temp_frames'
os.makedirs(temp_frames_folder, exist_ok=True)

# Используем ffmpeg для извлечения кадров из видео
subprocess.run(['ffmpeg', '-loglevel', 'quiet', '-i', video_path, f'{temp_frames_folder}/frame_%04d.png'], check=True)

# Подготовка для записи выходного видео с использованием ffmpeg
output_path = 'output_with_emotions.mp4'
output_frames_folder = 'output_frames'
os.makedirs(output_frames_folder, exist_ok=True)

# Получаем список всех кадров
frame_files = sorted(os.listdir(temp_frames_folder))

# Получение фреймрейта видео с использованием ffprobe
ffprobe_command = [
    'ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=r_frame_rate',
    '-of', 'default=noprint_wrappers=1:nokey=1', video_path
]
frame_rate_output = subprocess.check_output(ffprobe_command).decode().strip()
num, denom = map(int, frame_rate_output.split('/'))
frame_rate = num / denom

try:
    for frame_file in tqdm(frame_files, desc="Processing video frames"):
        # Загружаем кадр
        frame_path = os.path.join(temp_frames_folder, frame_file)
        frame = Image.open(frame_path)
        
        # Анализ эмоции и получение региона лица с помощью DeepFace
        emotions, region = analyze_emotion(np.array(frame))
        if emotions and region:
            x, y, w, h = region.get('x', 0), region.get('y', 0), region.get('w', 0), region.get('h', 0)
            dominant_emotion = max(emotions, key=emotions.get)
            box_text = "\n".join([f"{emotion}: {prob:.2f}%" for emotion, prob in emotions.items()])
            
            # Рисуем прямоугольник и текст с помощью Pillow
            draw = ImageDraw.Draw(frame)
            draw.rectangle([x, y, x + w, y + h], outline="green", width=2)
            try:
                font = ImageFont.truetype("DejaVuSans-Bold.ttf", size=16)  # Используем шрифт большего размера
            except IOError:
                font = ImageFont.load_default()  # Используем шрифт по умолчанию
            text_x = 10
            text_y = 10
            for line in box_text.split("\n"):
                color = "red" if line.startswith(dominant_emotion) else "green"
                draw.text((text_x, text_y), line, fill=color, font=font)
                text_y += 20  # Смещаем каждую строку ниже с увеличенным отступом
        
        # Сохраняем обработанный кадр
        output_frame_path = os.path.join(output_frames_folder, frame_file)
        frame.save(output_frame_path)

    # Кодирование выходного видео с помощью ffmpeg (перезапись существующего файла)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'quiet', '-framerate', str(frame_rate), '-i', f'{output_frames_folder}/frame_%04d.png', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', output_path], check=True)

finally:
    # Освобождение ресурсов и удаление временных файлов
    # Удаление временных папок и кадров
    pass
    #for folder in [temp_frames_folder, output_frames_folder]:
    #    for file in os.listdir(folder):
    #        os.remove(os.path.join(folder, file))
    #    os.rmdir(folder)


Processing video frames: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [01:09<00:00,  8.64it/s]
